In [ ]:
import os

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MaxNLocator
from matplotlib.backends.backend_pdf import PdfPages

from mg_sampling import PROJECT_DIR
from mg_sampling.constants import RT
from mg_sampling.tools import read_plumed, read_plumed_fes, range_to_edges, edges_to_range, finitemin, finitemax, block_analysis, read_wsv, demux
from mg_sampling.bias import bias_fn

LABELS = {
    'amber_tip3p': "Amber/TIP3P",
    'mMg_tip3p': "microMg/TIP3P",
    'nMg_tip3p': "nanoMg/TIP3P", 
    'mMg_opc': "microMg/OPC",
    'ecc_spce': "AmberECC/SPCE",
}

FORCEFIELDS_ARCHIVE = ['amber_tip3p', 'mMg_tip3p', 'nMg_tip3p', 'mMg_opc', 'ecc_spce']
FORCEFIELDS = ['mMg_tip3p']

os.makedirs('figures/plain_md', exist_ok=True)

In [ ]:
system = '7ez0'
ff = 'nMg_tip3p_hard'
nrep = 4
# if ff.endswith('bv'):
#     cv_dmx = np.stack([
#         read_plumed(PROJECT_DIR / f'replex/7ez0/{ff}/rep{i:02d}/bias.COLVAR.{i}').drop_duplicates('time').assign(rep=i)
#         for i in range(12)
#     ])[:, :, [0, 3, 1, 2, 5, 4]]
#     cv = demux(cv, repindex_file=PROJECT_DIR / f'replex/7ez0/{ff}/rep00/bias_index.xvg')
# else:
#     cv = np.stack([
#         read_plumed(PROJECT_DIR / f'replex/7ez0/{ff}/rep{i:02d}/bias.COLVAR.{i}').drop_duplicates('time').assign(rep=i)
#         for i in range(12)
#     ])
#     cv_dmx = demux(cv, repindex_file=PROJECT_DIR / f'replex/7ez0/{ff}/replica_index.xvg')

nox_dmx = np.stack([
    read_plumed(PROJECT_DIR / f'plain_md/{system}/{ff}/rep{i:02d}/cv.COLVAR.{i}').drop_duplicates('time').assign(rep=i)
    for i in range(nrep)
])
nox_dmx[:, :, 1:-2:3] *= 1.039661
nox_dmx[:, :, 2:-2:3] *= 1.039661

In [ ]:
# New bound ions configurations found
k = 0
dt = nox_dmx[k, 1, 0] / 1000 - nox_dmx[k, 0, 0] / 1000
fig, ax = plt.subplots()
is_bound_mg = (nox_dmx[k, :, 1:-2:3] > 0.5) * 1 + (nox_dmx[k, :, 1:-2:3] > 1.5) * 1 + (nox_dmx[k, :, 1:-2:3] > 2.5) * 1
from matplotlib.patches import Patch
cmap = plt.cm.gnuplot2_r
custom_patches = [Patch(color=cmap(0/3), label='Unbound'),
                Patch(color=cmap(1/3), label='Monodentate'),
                Patch(color=cmap(2/3), label='Bidentate'),
                Patch(color=cmap(3/3), label='> Bidentate'),]
ax.imshow(is_bound_mg.T, origin='lower', extent=[0, len(is_bound_mg) / 10, 0, is_bound_mg.shape[1]], aspect='auto', cmap=cmap, interpolation='auto')
ax.set_ylabel(r"Mg$^{2+}$ index")
ax.set_title(f"Replica {k}")
ax.legend(handles=custom_patches)
